# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hafsa-SE/flyrank-assignment/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding #4 -- The Freshness Multiplier** (growth-to-decline ratio by freshness window).
The paper reports 31-90 days as the strongest stable freshness band (7.88:1 growth-to-decline),
and is careful to flag that the `361+` bucket's 283:1 ratio is unstable because it rests on
just 1 declining page. That self-flagging is exactly right, and it's the same instinct I want
to apply to my own top-10 review.

*My methodology question:* the growing/declining label behind this ratio comes from
`trend_direction`, a 30-day-vs-previous-30-day comparison -- the same label my own Week-5
model predicts. A 30-day window is short enough that a single seasonal dip or a tracking gap
could flip a page's label without any real change in the page. Before treating "31-90 days is
the strongest window" as a scheduling rule, I'd ask: **does the growth/decline ratio hold up
if the comparison window is widened to 60 or 90 days, or does the 31-90 tier's advantage
shrink once short-window noise is averaged out?** The paper doesn't report that robustness
check, so I'd want to see it before I built a refresh calendar around exactly this window.

**Finding #10 -- AI Model Performance** (age-controlled cohort: OpenAI vs. Gemini health).
The paper deliberately controls for age (comparing model families inside the same publication
band) instead of the raw pooled comparison, and it explicitly declines to call a winner --
"Gemini leads some cohorts and OpenAI leads others." That restraint is a good model for how I
should talk about my own Week-5 Random Forest result.

*My methodology question:* age-controlling removes one confound, but content pieces aren't
randomly assigned to a model provider -- a client or workflow may have systematically chosen
one provider for one *kind* of content (e.g., a client using OpenAI mainly for informational
long-tail pages, another using Gemini mainly for transactional pages). **Does the
age-controlled comparison also control for client and intent mix, or could an uneven provider
x client x intent distribution still be driving part of the health-score gap inside each age
cohort?** Without that check, "Gemini leads some cohorts" could partly be "certain clients'
content leads some cohorts" wearing a model-family label.


In [1]:
# No computation needed for this section -- it's a close reading of the paper's own tables.
# Quick anchor: confirm the shared label (trend_direction) is the same one Finding #4 uses.
import pandas as pd
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(df["trend_direction"].value_counts())
print("\nThis is the same 30d-vs-prev-30d trend label the paper's Finding #4 growth/decline ratio")
print("is built on, and the same label my own Week-5 model (is_declining_label) predicts.")


trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

This is the same 30d-vs-prev-30d trend label the paper's Finding #4 growth/decline ratio
is built on, and the same label my own Week-5 model (is_declining_label) predicts.


## 2. My model under an honest split (before/after)

**Before:** a plain random 80/20 row split (ignores `client_id` -- rows from the same client
can land in both train and test, so the model can partly memorize per-client quirks).
**After:** the client-grouped split I actually used in Week 5 (held-out clients never seen in
training). Same features, same models (Logistic Regression, Random Forest), same metric
(precision@20/@50), same random seed -- only the split changes.


In [2]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier",
    "impression_tier", "position_tier",
]
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{col}"] = np.log1p(df[col].clip(lower=0))
NUMERIC_FEATURES += ["log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d"]

numeric_frame = df[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
categorical_frame = df[CATEGORICAL_FEATURES].fillna("unknown").astype(str)
encoded = pd.get_dummies(categorical_frame, prefix=CATEGORICAL_FEATURES, dtype=float)
X = pd.concat([numeric_frame.reset_index(drop=True), encoded.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].astype(int)

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    top = np.asarray(y_true)[order[:k]]
    return float(top.mean()) if len(top) else float("nan")

def fit_and_score(train_idx, test_idx, label):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    rows = []
    for name, model in {
        "logistic_regression": Pipeline([("scaler", StandardScaler()),
                                          ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))]),
        "random_forest": RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
                                                 class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE),
    }.items():
        model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]
        rows.append({
            "split": label, "model": name,
            "precision_at_20": precision_at_k(y_test, proba, 20),
            "precision_at_50": precision_at_k(y_test, proba, 50),
            "n_test": len(y_test),
        })
    return rows

# --- BEFORE: plain random row split, ignores client_id ---
rng = np.random.default_rng(RANDOM_STATE)
shuffled_rows = rng.permutation(len(df))
n_test_rows = int(round(len(df) * 0.2))
random_test_idx = shuffled_rows[:n_test_rows]
random_train_idx = shuffled_rows[n_test_rows:]
overlap_clients = set(df["client_id"].iloc[random_train_idx]) & set(df["client_id"].iloc[random_test_idx])
print(f"BEFORE (random row split): {len(overlap_clients)} of {df['client_id'].nunique()} clients appear in BOTH train and test.")

# --- AFTER: client-grouped split (same design as Week 5) ---
client_series = df["client_id"].astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()
rng2 = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng2.permutation(unique_clients)
n_test_clients = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:n_test_clients])
grouped_test_mask = client_series.isin(test_clients).to_numpy()
grouped_train_idx = np.where(~grouped_test_mask)[0]
grouped_test_idx = np.where(grouped_test_mask)[0]
print(f"AFTER (client-grouped split): 0 of {df['client_id'].nunique()} clients appear in both (by construction).")

results = fit_and_score(random_train_idx, random_test_idx, "BEFORE: random row split") + \
          fit_and_score(grouped_train_idx, grouped_test_idx, "AFTER: client-grouped split")
comparison = pd.DataFrame(results).set_index(["split", "model"])
print("\n" + comparison.round(3).to_string())
print("\nThe random split lets a model partly recognize clients it already saw in training, so its")
print("numbers read more optimistic than the grouped split's -- the grouped numbers are the honest ones.")


BEFORE (random row split): 31 of 32 clients appear in BOTH train and test.
AFTER (client-grouped split): 0 of 32 clients appear in both (by construction).



                                                 precision_at_20  precision_at_50  n_test
split                       model                                                        
BEFORE: random row split    logistic_regression             0.90             0.92    6000
                            random_forest                   1.00             0.98    6000
AFTER: client-grouped split logistic_regression             0.35             0.40    2325
                            random_forest                   0.90             0.78    2325

The random split lets a model partly recognize clients it already saw in training, so its
numbers read more optimistic than the grouped split's -- the grouped numbers are the honest ones.


## 3. Leakage audit

Same hunt as Week 3/5, applied to the final feature set: forbidden-column check, plus a
correlation scan for any feature that's suspiciously close to the label itself (a near-1.0
correlation would mean the feature is functionally leaking the label).


In [3]:
FORBIDDEN = {"trend_direction", "trend_pct", "is_declining_label",
             "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
             "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"}
used = set(NUMERIC_FEATURES + CATEGORICAL_FEATURES)
overlap = used & FORBIDDEN
print(f"Forbidden columns used as features (should be empty): {sorted(overlap)}")
assert not overlap, "Leakage: a forbidden/label-derived column is in the feature list."

# Correlation of every numeric feature with the label -- flag anything suspiciously strong
numeric_corr = numeric_frame.assign(is_declining_label=y).corr(numeric_only=True)["is_declining_label"].drop("is_declining_label")
numeric_corr = numeric_corr.sort_values(key=abs, ascending=False)
print("\nNumeric feature correlation with label (sorted by |r|):")
print(numeric_corr.round(3).to_string())

suspicious = numeric_corr[numeric_corr.abs() > 0.9]
print(f"\nFeatures with |correlation| > 0.9 with the label (should be empty): {suspicious.index.tolist()}")
assert suspicious.empty, "Leakage: a feature is near-perfectly correlated with the label."
print("\nHighest correlations are all well under 0.9 and match domain sense (position/visibility-shaped),")
print("not a single feature carrying the label. No leakage found in this feature set.")


Forbidden columns used as features (should be empty): []

Numeric feature correlation with label (sorted by |r|):
days_with_impressions     0.190
log_impressions_90d       0.177
content_age_days         -0.164
word_count                0.119
char_count                0.108
days_since_last_update    0.081
ctr                      -0.062
avg_position             -0.029
days_with_sessions       -0.025
log_sessions_90d          0.015
search_volume            -0.014
engagement_rate          -0.013
competition               0.013
cpc                      -0.006
log_ai_sessions_90d      -0.004
log_clicks_90d            0.003
scroll_rate              -0.003
ai_traffic_pct            0.002

Features with |correlation| > 0.9 with the label (should be empty): []

Highest correlations are all well under 0.9 and match domain sense (position/visibility-shaped),
not a single feature carrying the label. No leakage found in this feature set.


## 4. Claim rewrite

Three of my own boldest sentences from Week 4/5, rewritten in safe language.


In [4]:
claims = [
    {
        "original": "Random Forest beats the baseline.",
        "rewrite": (
            "In this one client-grouped test split (6 held-out clients), Random Forest showed "
            "higher measured precision@50 than the Week-4 rule-based baseline (0.78 vs 0.66). "
            "This is an observed, directional result on one split -- decision-support for "
            "prioritizing the model over the rule, not a guarantee it will hold on a different "
            "client mix or a different time period."
        ),
    },
    {
        "original": "CTR clearly drops as position gets worse.",
        "rewrite": (
            "Weighted CTR was measured to decrease monotonically from top_3 (0.49%) to deep "
            "(0.04%) across all five position tiers in this portfolio slice (n>=1,116 per tier). "
            "This is an observed pattern in this dataset's 90-day window, consistent with the "
            "paper's own Finding #3 -- it supports prioritizing page-1 CTR fixes over deep-tier "
            "pages as a directional bet, not a universal CTR law."
        ),
    },
    {
        "original": "The near-zero-CTR pages the model got wrong are a technical issue, not a content issue.",
        "rewrite": (
            "The three highest-confidence misses in the held-out test set were all top_3-position "
            "pages with 1-3 impressions_90d and near-zero CTR -- a pattern consistent with a "
            "technical or indexing issue rather than a content gap, but this is a hypothesis "
            "from 3 examples, not a confirmed diagnosis. It's decision-support for a technical "
            "check, not a finding to publish as a portfolio-wide claim."
        ),
    },
]

for i, c in enumerate(claims, 1):
    print(f"--- Claim {i} ---")
    print(f"ORIGINAL: {c['original']}")
    print(f"REWRITE:  {c['rewrite']}")
    print()


--- Claim 1 ---
ORIGINAL: Random Forest beats the baseline.
REWRITE:  In this one client-grouped test split (6 held-out clients), Random Forest showed higher measured precision@50 than the Week-4 rule-based baseline (0.78 vs 0.66). This is an observed, directional result on one split -- decision-support for prioritizing the model over the rule, not a guarantee it will hold on a different client mix or a different time period.

--- Claim 2 ---
ORIGINAL: CTR clearly drops as position gets worse.
REWRITE:  Weighted CTR was measured to decrease monotonically from top_3 (0.49%) to deep (0.04%) across all five position tiers in this portfolio slice (n>=1,116 per tier). This is an observed pattern in this dataset's 90-day window, consistent with the paper's own Finding #3 -- it supports prioritizing page-1 CTR fixes over deep-tier pages as a directional bet, not a universal CTR law.

--- Claim 3 ---
ORIGINAL: The near-zero-CTR pages the model got wrong are a technical issue, not a content iss

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.